### Check GPU availability and display GPU information

In [1]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)


Tue Mar 10 09:52:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   49C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [2]:
%%capture
!pip install wget
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate


### Install system dependencies and clone the KenLM language model toolkit

In [3]:
!sudo apt-get update
!sudo apt-get install build-essential libboost-all-dev cmake zlib1g-dev libbz2-dev liblzma-dev
!sudo apt-get install libboost-all-dev libeigen3-dev
!git clone https://github.com/kpu/kenlm
%cd kenlm
!pip install e .

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,431 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

### Build and install KenLM from source

In [4]:
!mkdir build
%cd build
!cmake ..
!make -j 4
!sudo make install

mkdir: cannot create directory ‘build’: File exists
/content/kenlm/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.74.0/BoostConfig.cmake (found suitable v

### Add KenLM binaries to the system PATH and verify installation

In [5]:
import os
os.environ['PATH'] += ":/content/kenlm/build/bin"  # Use your exact path
!echo $PATH  # Verify
!which lmplz  # Test


/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/kenlm/build/bin
/usr/local/bin/lmplz


### Authenticate with Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


### Load the Sinhala ASR dataset with specified train/test splits

In [6]:
from datasets import load_dataset

common_voice_train = load_dataset(
    "keshan/large-sinhala-asr-dataset",
    "si",
    split="train[:40%]",
    trust_remote_code=True,
)

common_voice_test = load_dataset(
    "keshan/large-sinhala-asr-dataset",
    "si",
    split="test[:40%]",
    trust_remote_code=True,
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


large-sinhala-asr-dataset.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

### Inspect the training dataset structure and size

In [7]:
common_voice_train

Dataset({
    features: ['filename', 'x', 'sentence', 'file'],
    num_rows: 53018
})

### Inspect the test dataset structure and size

In [8]:
common_voice_test

Dataset({
    features: ['filename', 'x', 'sentence', 'file'],
    num_rows: 9356
})

### Rename the 'file' column to 'audio' and drop unused columns for downstream compatibility

In [9]:
# Rename the audio-path column to match the rest of the notebook
common_voice_train = common_voice_train.rename_column("file", "audio")
common_voice_test  = common_voice_test.rename_column("file", "audio")

# Drop columns that are not used downstream (dataset-specific)
common_voice_train = common_voice_train.remove_columns(["filename", "x"])
common_voice_test  = common_voice_test.remove_columns(["filename", "x"])

from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

# Select the 'train' split before calling show_random_elements
show_random_elements(common_voice_train.remove_columns(["audio"]))


,sentence
0,මොරගහකන්ද ජලාශය
1,මගේ දැඩි අවධානයට යොමු වුණා.
2,බල පෑ එකම කාරණය
3,සමාජයේ සුබ සිද්ධිය උදෙසා
4,පසුව පොලිස් කණ්ඩායමක් සමග ජැක්ව අත්අඩංගුවට ගන්න ගියා
5,අඩුගානෙ ස්වර්ගයට යන්න හිතන් ඉන්න අයවත්
6,දුකක් තරහක්
7,දහවලේදී නීතීඥයෙක් ලෙස කටයුතු කරමින්
8,දාහක් දෙදාහක් දුන්නත් කොච්චර දෙයක්ද
9,මාලිගාවට අයත්


### Define and apply text normalization: remove punctuation, Latin characters, digits, and normalize Unicode for Sinhala text

In [10]:
import re
import unicodedata

# punctuation list (kept in the same style as your notebook)
# added Sinhala/Indic danda + Sinhala punctuation marks
chars_to_remove_regex = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\(\)\'\”\‘‘\’\ʻ\”\“\–\—\…\»\«।॥෴]'

def remove_punctuation(batch):
    text = batch["sentence"]
    text = unicodedata.normalize("NFC", text)

    # remove punctuation
    text = re.sub(chars_to_remove_regex, "", text)

    # remove latin letters + digits (Sinhala baseline)
    text = re.sub(r"[A-Za-z0-9]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    batch["sentence"] = text
    return batch

common_voice_train = common_voice_train.map(remove_punctuation)
common_voice_test = common_voice_test.map(remove_punctuation)

# Keep only Sinhala Unicode block + spaces (Sinhala: U+0D80–U+0DFF)
sinhala_allowed = re.compile(r"^[\u0D80-\u0DFF\s]+$")

def keep_only_sinhala_rows(batch):
    s = batch["sentence"].strip()
    if len(s) == 0:
        return False
    return bool(sinhala_allowed.match(s))

common_voice_train = common_voice_train.filter(keep_only_sinhala_rows)
common_voice_test = common_voice_test.filter(keep_only_sinhala_rows)

Map:   0%|          | 0/53018 [00:00<?, ? examples/s]

Map:   0%|          | 0/9356 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53018 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9356 [00:00<?, ? examples/s]

### Extract the unique character vocabulary from train and test sets

In [11]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

{' ': 0, 'ං': 1, 'ඃ': 2, 'අ': 3, 'ආ': 4, 'ඇ': 5, 'ඈ': 6, 'ඉ': 7, 'ඊ': 8, 'උ': 9, 'ඌ': 10, 'ඍ': 11, 'එ': 12, 'ඒ': 13, 'ඓ': 14, 'ඔ': 15, 'ඕ': 16, 'ඖ': 17, 'ක': 18, 'ඛ': 19, 'ග': 20, 'ඝ': 21, 'ඞ': 22, 'ඟ': 23, 'ච': 24, 'ඡ': 25, 'ජ': 26, 'ඣ': 27, 'ඤ': 28, 'ඥ': 29, 'ට': 30, 'ඨ': 31, 'ඩ': 32, 'ඪ': 33, 'ණ': 34, 'ඬ': 35, 'ත': 36, 'ථ': 37, 'ද': 38, 'ධ': 39, 'න': 40, 'ඳ': 41, 'ප': 42, 'ඵ': 43, 'බ': 44, 'භ': 45, 'ම': 46, 'ඹ': 47, 'ය': 48, 'ර': 49, 'ල': 50, 'ව': 51, 'ශ': 52, 'ෂ': 53, 'ස': 54, 'හ': 55, 'ළ': 56, 'ෆ': 57, '්': 58, 'ා': 59, 'ැ': 60, 'ෑ': 61, 'ි': 62, 'ී': 63, 'ු': 64, 'ූ': 65, 'ෘ': 66, 'ෙ': 67, 'ේ': 68, 'ෛ': 69, 'ො': 70, 'ෝ': 71, 'ෞ': 72, 'ෟ': 73, 'ෲ': 74, 'ෳ': 75}


### Remove all non-Sinhala characters from transcriptions and re-extract the character vocabulary

In [12]:
import re

def remove_non_sinhala_characters(batch):
    # remove anything that is NOT Sinhala block or whitespace
    batch["sentence"] = re.sub(r"[^\u0D80-\u0DFF\s]", "", batch["sentence"])
    batch["sentence"] = re.sub(r"\s+", " ", batch["sentence"]).strip()
    return batch

common_voice_train = common_voice_train.map(remove_non_sinhala_characters)
common_voice_test = common_voice_test.map(remove_non_sinhala_characters)


# extract unique characters again
vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

{' ': 0, 'ං': 1, 'ඃ': 2, 'අ': 3, 'ආ': 4, 'ඇ': 5, 'ඈ': 6, 'ඉ': 7, 'ඊ': 8, 'උ': 9, 'ඌ': 10, 'ඍ': 11, 'එ': 12, 'ඒ': 13, 'ඓ': 14, 'ඔ': 15, 'ඕ': 16, 'ඖ': 17, 'ක': 18, 'ඛ': 19, 'ග': 20, 'ඝ': 21, 'ඞ': 22, 'ඟ': 23, 'ච': 24, 'ඡ': 25, 'ජ': 26, 'ඣ': 27, 'ඤ': 28, 'ඥ': 29, 'ට': 30, 'ඨ': 31, 'ඩ': 32, 'ඪ': 33, 'ණ': 34, 'ඬ': 35, 'ත': 36, 'ථ': 37, 'ද': 38, 'ධ': 39, 'න': 40, 'ඳ': 41, 'ප': 42, 'ඵ': 43, 'බ': 44, 'භ': 45, 'ම': 46, 'ඹ': 47, 'ය': 48, 'ර': 49, 'ල': 50, 'ව': 51, 'ශ': 52, 'ෂ': 53, 'ස': 54, 'හ': 55, 'ළ': 56, 'ෆ': 57, '්': 58, 'ා': 59, 'ැ': 60, 'ෑ': 61, 'ි': 62, 'ී': 63, 'ු': 64, 'ූ': 65, 'ෘ': 66, 'ෙ': 67, 'ේ': 68, 'ෛ': 69, 'ො': 70, 'ෝ': 71, 'ෞ': 72, 'ෟ': 73, 'ෲ': 74, 'ෳ': 75}


### Finalize the vocabulary: replace space with pipe delimiter, add [UNK] and [PAD] tokens, and save vocab.json

In [13]:
# use "|" as word delimiter instead of space
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocab size:", len(vocab_dict))

import json
with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


Vocab size: 78


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [14]:
from transformers import Wav2Vec2CTCTokenizer
from transformers import SeamlessM4TFeatureExtractor
from transformers import Wav2Vec2BertProcessor

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


### Resample all audio to 16kHz and preview a random training sample

In [15]:
from datasets import Audio

common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

print(common_voice_train[0]["audio"])

import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train) - 1)
print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

ipd.Audio(
    data=common_voice_train[rand_int]["audio"]["array"],
    autoplay=True,
    rate=16000,
)


{'path': '/root/.cache/huggingface/datasets/downloads/extracted/asr_sinhala/data/bd/bd535bed99.flac', 'array': array([-0.00027466,  0.00024414, -0.0007019 , ...,  0.00106812,
        0.00085449,  0.00024414]), 'sampling_rate': 16000}
Target text: බාන්ඩයකට එක් කලාම
Input array shape: (68800,)
Sampling rate: 16000


### Write all transcriptions to a text file for KenLM language model training

In [16]:
with open("lm_corpus.txt", "w", encoding="utf-8") as f:
    for ex in common_voice_train:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")
    for ex in common_voice_test:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")

print("Wrote LM corpus to lm_corpus.txt")


Wrote LM corpus to lm_corpus.txt


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

common_voice_train = common_voice_train.map(
    prepare_dataset,
    remove_columns=common_voice_train.column_names,
)
common_voice_test = common_voice_test.map(
    prepare_dataset,
    remove_columns=common_voice_test.column_names,
)


Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

### Define a custom data collator that pads input features and labels for CTC training

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
import evaluate

from transformers import Wav2Vec2BertProcessor

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

### Train a trigram KenLM language model from the corpus

In [ ]:
!lmplz -o 3 < lm_corpus.txt > lm.arpa

### Install pyctcdecode library for CTC beam search decoding with language model support

In [ ]:
!pip install pyctcdecode

### Build a CTC beam search decoder with KenLM language model integration

In [22]:
from pyctcdecode import build_ctcdecoder

# get labels in ID order
vocab = processor.tokenizer.get_vocab()  # token -> id
id2token = {v: k for k, v in vocab.items()}
labels = [id2token[i] for i in range(len(id2token))]

# path to your trained KenLM model (arpa or binary)
kenlm_model_path = "lm.arpa"  # <-- change if you use another path/name

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_model_path,
    alpha=0.5,  # tune on dev
    beta=1.0  # tune on dev
)


### Define the evaluation metrics function using LM-aided beam search decoding to compute WER and CER

In [29]:
import evaluate
import torch

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    # Convert logits to probabilities
    probs = torch.softmax(torch.tensor(pred_logits), dim=-1).numpy()

    # Decode with LM
    pred_str = [decoder.decode(p, beam_width=32) for p in probs]

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Mount Google Drive for persistent storage

In [25]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Run inference on the test set using LM-aided beam search decoding and compute WER/CER

In [30]:
import numpy as np
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor # Added AutoModelForCTC import
import torch

checkpoint_dir = "/content/drive/MyDrive/final_model35"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

model.eval()

all_pred_raw = []        # ASR + LM only
all_pred_corrected = []  # ASR + LM + T5 spell corrector
all_refs = []            # ground-truth text

for i, ex in enumerate(common_voice_test):
    # 1. Forward pass on stored input_features from your prepared dataset
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(input_feats).logits

    # 2. Convert logits to probabilities
    probs = torch.softmax(logits[0], dim=-1).cpu().numpy()

    # 3. Decode with LM (wav2vec2 + CTC + KenLM)
    pred_text = decoder.decode(probs)
    all_pred_raw.append(pred_text)

    # 5. Reference text from labels
    ref_text = processor.decode(ex["labels"], group_tokens=False).lower()
    all_refs.append(ref_text)

# 6. Compute WERs
wer_raw = wer_metric.compute(
    predictions=all_pred_raw,
    references=all_refs,
)
cer_raw = cer_metric.compute(predictions=all_pred_raw, references=all_refs)

print(f"WER (ASR + LM only)           : {wer_raw:.4f}")
print(f"CER (ASR + LM only)           : {cer_raw:.4f}")

# Optional: inspect a few examples
for idx in range(3):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("RAW :", all_pred_raw[idx])

WER (ASR + LM only)           : 0.0700
CER (ASR + LM only)           : 0.0189

--- Example 0 ---
REF : ආචාර්යවරිය නම් ලකුණු කරන අතරතුර
RAW : ආචාර්යවරිය නම් ලකුණු කරන අතරතුර

--- Example 1 ---
REF : මේ අයුරින් මරණයට පත් වේ
RAW : මේ අයුරින් මරණයට පත් වේ

--- Example 2 ---
REF : ඒකට මොකද මේකෙන්
RAW : ඒකට මොකද මේකෙන්
